In [1]:
# =========================
# Final model comparison:
# L1 Logistic, L2 Logistic, Random Forest, XGBoost
# on finance-only / text-only / combined
# =========================

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# If xgboost is not installed, run:
# pip install xgboost
from xgboost import XGBClassifier


# =========================
# 1. Load merged modeling datasets
# =========================
FIN_PATH = "model_finance_only.csv"
TEXT_PATH = "model_text_only.csv"
COMBINED_PATH = "model_finance_text_combined.csv"

df_fin = pd.read_csv(FIN_PATH)
df_text = pd.read_csv(TEXT_PATH)
df_combined = pd.read_csv(COMBINED_PATH)

print("df_fin shape:", df_fin.shape)
print("df_text shape:", df_text.shape)
print("df_combined shape:", df_combined.shape)


# =========================
# 2. Define feature columns
# =========================
# Finance features: PC1 ~ PC8
finance_feature_cols = [c for c in df_fin.columns if c.startswith("PC")]

# Text features: exclude IDs, labels, and highly redundant columns
drop_text_cols = {
    "companyid", "companyname", "company_name", "companyid_bridge",
    "ticker", "label", "label_binary", "title",
    "pct_positive", "pct_negative", "pct_neutral",
    "mean_p_positive", "mean_p_negative", "mean_p_neutral",
    "ai_keyword_count"
}
text_feature_cols = [
    c for c in df_text.columns
    if c not in drop_text_cols and not c.startswith("PC")
]

# Combined features: keep finance PCs + cleaned text features
drop_combined_cols = {
    "companyid", "companyname", "company_name", "companyid_bridge",
    "ticker", "label", "label_binary", "title",
    "tic", "gvkey", "datadate",
    "pct_positive", "pct_negative", "pct_neutral",
    "mean_p_positive", "mean_p_negative", "mean_p_neutral",
    "ai_keyword_count"
}
combined_feature_cols = [
    c for c in df_combined.columns
    if c not in drop_combined_cols
]

print("\nFinance features:", finance_feature_cols)
print("\nText features:", text_feature_cols)
print("\nCombined features:", combined_feature_cols)


# =========================
# 3. Organize datasets
# =========================
datasets = {
    "X_fin_only": (df_fin, finance_feature_cols),
    "X_text_only": (df_text, text_feature_cols),
    "X_fin_plus_X_text": (df_combined, combined_feature_cols),
}


# =========================
# 4. Define models
# =========================
def build_model(model_name):
    if model_name == "logit_l1":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l1",
                solver="saga",
                C=1.0,
                max_iter=5000,
                class_weight="balanced",
                random_state=42
            ))
        ])

    elif model_name == "logit_l2":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                C=1.0,
                max_iter=5000,
                class_weight="balanced",
                random_state=42
            ))
        ])

    elif model_name == "random_forest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=3,
                class_weight="balanced",
                random_state=42
            ))
        ])

    elif model_name == "xgboost":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                n_estimators=300,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=1.0,
                random_state=42,
                eval_metric="logloss",
                use_label_encoder=False
            ))
        ])

    else:
        raise ValueError(f"Unknown model_name: {model_name}")


model_names = [
    "logit_l1",
    "logit_l2",
    "random_forest",
    "xgboost"
]


# =========================
# 5. Evaluation function
# =========================
def evaluate_model(df, feature_cols, model_name):
    X = df[feature_cols].copy()
    y = df["label_binary"].copy()

    model = build_model(model_name)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "accuracy": "accuracy",
        "f1": "f1",
        "roc_auc": "roc_auc"
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    return {
        "model_name": model_name,
        "n_samples": len(df),
        "n_features": len(feature_cols),
        "accuracy_mean": np.mean(scores["test_accuracy"]),
        "accuracy_std": np.std(scores["test_accuracy"]),
        "f1_mean": np.mean(scores["test_f1"]),
        "f1_std": np.std(scores["test_f1"]),
        "roc_auc_mean": np.mean(scores["test_roc_auc"]),
        "roc_auc_std": np.std(scores["test_roc_auc"]),
    }


# =========================
# 6. Run all experiments
# =========================
results = []

for feature_set_name, (df_current, feature_cols) in datasets.items():
    print(f"\nRunning models for: {feature_set_name}")
    print(f"Samples = {len(df_current)}, Features = {len(feature_cols)}")

    for model_name in model_names:
        print(f"  -> {model_name}")
        out = evaluate_model(df_current, feature_cols, model_name)
        out["feature_set"] = feature_set_name
        results.append(out)

results_df = pd.DataFrame(results)

# Reorder columns
results_df = results_df[
    [
        "feature_set",
        "model_name",
        "n_samples",
        "n_features",
        "accuracy_mean",
        "accuracy_std",
        "f1_mean",
        "f1_std",
        "roc_auc_mean",
        "roc_auc_std",
    ]
]

print("\n=== Final model comparison ===")
print(results_df.round(4).to_string(index=False))


# =========================
# 7. Save results
# =========================
results_df.to_csv("model_results.csv", index=False)

print("\nSaved file:")
print("  model_results.csv")

df_fin shape: (158, 14)
df_text shape: (154, 33)
df_combined shape: (154, 43)

Finance features: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8']

Text features: ['n_sentences', 'mean_sentiment', 'std_sentiment', 'ratio_strong_negative', 'ratio_strong_positive', 'topic_00_energy_utilities', 'topic_01_advertising_marketing', 'topic_02_ai_semiconductor_design', 'topic_03_ai_enterprise_healthcare_services', 'topic_04_financial_reporting_geographic', 'topic_05_financial_reporting_accounting', 'topic_06_government_defense', 'topic_07_project_execution_capital', 'topic_08_china_ai_semiconductor_supply_chain', 'topic_09_ai_cloud_infrastructure', 'topic_10_chinese_gaming_streaming', 'topic_11_ai_saas_cybersecurity', 'ai_topic_loading_sum', 'ai_keyword_ratio']

Combined features: ['n_sentences', 'mean_sentiment', 'std_sentiment', 'ratio_strong_negative', 'ratio_strong_positive', 'topic_00_energy_utilities', 'topic_01_advertising_marketing', 'topic_02_ai_semiconductor_design', 'topic_03_

/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [20:22:49] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [20:22:49] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [20:22:49] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [20:22:49] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i

  -> random_forest
  -> xgboost


/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [20:22:50] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [20:22:50] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [20:22:50] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [20:22:50] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i


Running models for: X_fin_plus_X_text
Samples = 154, Features = 27
  -> logit_l1
  -> logit_l2
  -> random_forest
  -> xgboost


/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [20:22:51] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [20:22:51] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [20:22:51] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [20:22:51] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i


=== Final model comparison ===
      feature_set    model_name  n_samples  n_features  accuracy_mean  accuracy_std  f1_mean  f1_std  roc_auc_mean  roc_auc_std
       X_fin_only      logit_l1        158           8         0.5629        0.0846   0.5186  0.1078        0.5672       0.0746
       X_fin_only      logit_l2        158           8         0.5633        0.0782   0.5165  0.0840        0.5617       0.0682
       X_fin_only random_forest        158           8         0.5306        0.0792   0.5294  0.0828        0.5680       0.0558
       X_fin_only       xgboost        158           8         0.5183        0.0483   0.4953  0.0722        0.5396       0.0481
      X_text_only      logit_l1        154          19         0.7462        0.0682   0.7533  0.0692        0.7889       0.0365
      X_text_only      logit_l2        154          19         0.7333        0.0579   0.7421  0.0521        0.7821       0.0384
      X_text_only random_forest        154          19         0.7271   